# Fine-tune YOLOv8n on car keys + card

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (this notebook asks for one, but double-check).
2. Have your labeled dataset exported from Roboflow as a **.zip** in **YOLOv8** format.

Then run the cells top to bottom.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install Ultralytics

In [ ]:
%pip install -q -U ultralytics

## 3. Upload the dataset
Pick the Roboflow `.zip` when the upload button appears.

In [ ]:
import os, glob, zipfile
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))

DATASET_DIR = "/content/dataset"
os.makedirs(DATASET_DIR, exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall(DATASET_DIR)

print(os.listdir(DATASET_DIR))

## 4. Fix the dataset config
Roboflow's `data.yaml` uses relative paths that often break in Colab, so rewrite them as paths from the dataset root.

In [ ]:
import yaml

yaml_path = glob.glob(f"{DATASET_DIR}/**/data.yaml", recursive=True)[0]
root = os.path.dirname(yaml_path)

with open(yaml_path) as f:
    data = yaml.safe_load(f)

data["path"] = root
data["train"] = "train/images"
if os.path.isdir(f"{root}/valid/images"):
    data["val"] = "valid/images"
else:
    data["val"] = "train/images"
    print("WARNING: no validation split found, validating on training images (metrics will look too good).")
if os.path.isdir(f"{root}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)

with open(yaml_path, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("Classes:", data["names"])
for split in ["train", "valid", "test"]:
    d = f"{root}/{split}/images"
    if os.path.isdir(d):
        print(f"{split}: {len(os.listdir(d))} images")

## 5. Train
- `epochs=100` with `patience=20`: stops early if validation stops improving for 20 epochs.
- `imgsz=640`: keep this high, keys are small objects.
- If you run out of GPU memory, drop `batch` to 8.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project="/content/runs",
    name="keys_card",
    exist_ok=True,
)

RUN_DIR = str(model.trainer.save_dir)
print("Run saved to", RUN_DIR)

## 6. Check the results
mAP50 is the headline number: above ~0.8 is good for a two-class custom dataset.

In [ ]:
from IPython.display import Image, display

best = YOLO(f"{RUN_DIR}/weights/best.pt")
metrics = best.val(data=yaml_path)
print(f"mAP50: {metrics.box.map50:.3f}   mAP50-95: {metrics.box.map:.3f}")

display(Image(f"{RUN_DIR}/results.png", width=900))
display(Image(f"{RUN_DIR}/confusion_matrix.png", width=600))

## 7. Preview predictions on a few validation images

In [ ]:
samples = sorted(glob.glob(os.path.join(root, data["val"], "*")))[:4]
best.predict(samples, conf=0.5, save=True, project="/content/runs", name="preview", exist_ok=True)

for p in sorted(glob.glob("/content/runs/preview/*")):
    display(Image(p, width=500))

## 8. Export to ONNX for OpenCV
`opset=12` and a fixed input size keep it compatible with `cv2.dnn`.

In [ ]:
onnx_path = best.export(format="onnx", imgsz=640, opset=12, simplify=True, dynamic=False)
print(onnx_path)

## 9. Download
Bundles `best.onnx` (for `cv2.dnn`), `best.pt` (to retrain or use with Ultralytics), and `classes.txt` (class names in model order) into one zip.

In [ ]:
import shutil

OUT_DIR = "/content/export"
os.makedirs(OUT_DIR, exist_ok=True)
shutil.copy(onnx_path, OUT_DIR)
shutil.copy(f"{RUN_DIR}/weights/best.pt", OUT_DIR)
with open(f"{OUT_DIR}/classes.txt", "w") as f:
    for i in sorted(best.names):
        f.write(best.names[i] + "\n")

shutil.make_archive("/content/keys_card_model", "zip", OUT_DIR)
files.download("/content/keys_card_model.zip")